# 4. Validação referencial

Confere se todo ingrediente referenciado em recipes/buildings existe de
fato na tabela de items (direto ou via correção de casing). Um órfão aqui
vira um ingrediente que "desaparece" silenciosamente no app.


In [1]:
import json
from pathlib import Path

import pandas as pd

# Notebook lives in notebooks/etl/, so the repo root is two levels up.
REPO_ROOT = Path("../..").resolve()

ROOT = REPO_ROOT / "data/Pal/Content"
PAL = ROOT / "Pal"
ITEM_DT = PAL / "DataTable/Item/DT_ItemDataTable_Common.json"
RECIPE_DT = PAL / "DataTable/Item/DT_ItemRecipeDataTable_Common.json"
BUILDOBJECT_DT = PAL / "DataTable/MapObject/Building/DT_BuildObjectDataTable_Common.json"
BENCH_RECIPES = REPO_ROOT / "src/data/bench_recipes.json"
NAMES_DT_EN = ROOT / "L10N/en/Pal/DataTable/Text/DT_ItemNameText_Common.json"
NAMES_DT_PT_BR = ROOT / "L10N/pt-BR/Pal/DataTable/Text/DT_ItemNameText_Common.json"


def load_rows(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)[0]["Rows"]


In [2]:
items_raw = load_rows(ITEM_DT)
recipes_raw = load_rows(RECIPE_DT)
buildings_raw = load_rows(BUILDOBJECT_DT)

items_df = pd.DataFrame.from_dict(items_raw, orient="index")
recipes_df = pd.DataFrame.from_dict(recipes_raw, orient="index")
buildings_df = pd.DataFrame.from_dict(buildings_raw, orient="index")

print(f"items: {items_df.shape}, recipes: {recipes_df.shape}, buildings: {buildings_df.shape}")


items: (2466, 53), recipes: (1414, 20), buildings: (498, 32)


In [3]:
# Padroniza a string sentinela "None" (usada pela UE pra ausência de valor)
# para o None real do Python, e monta um resolvedor de id case-insensitive -
# ver 02_limpeza.ipynb pro raciocínio completo por trás disso.
items_clean = items_df.replace("None", None)
recipes_clean = recipes_df.replace("None", None)
buildings_clean = buildings_df.replace("None", None)

items_by_lower = {item_id.lower(): item_id for item_id in items_clean.index}


def resolve_item_id(raw_id):
    # pandas' .replace("None", None) upcasts these cells to NaN (a float),
    # not Python None - "is None" or plain truthiness checks silently miss
    # it and .lower() blows up on a float. pd.isna() catches both.
    if pd.isna(raw_id):
        return None
    if raw_id in items_clean.index:
        return raw_id
    return items_by_lower.get(raw_id.lower())


## Órfãos em recipes


In [4]:
rows = []
for product_id, recipe in recipes_clean.iterrows():
    product_count = recipe["Product_Count"] or 1
    for i in range(1, 6):
        material_id = recipe[f"Material{i}_Id"]
        material_count = recipe[f"Material{i}_Count"] or 0
        if pd.isna(material_id) or material_count == 0:
            continue
        rows.append({
            "item_id": product_id,
            "ingredient_id_raw": material_id,
            "ingredient_id": resolve_item_id(material_id),
        })

ingredients_long = pd.DataFrame(rows)
orphans_recipes = ingredients_long[ingredients_long["ingredient_id"].isna()]
print(f"{len(orphans_recipes)} referência(s) de recipe que não resolvem para nenhum item conhecido:")
orphans_recipes[["item_id", "ingredient_id_raw"]]


0 referência(s) de recipe que não resolvem para nenhum item conhecido:


,item_id,ingredient_id_raw


## Órfãos em buildings


In [5]:
building_rows = []
for building_id, building in buildings_clean.iterrows():
    for i in range(1, 5):
        material_id = building[f"Material{i}_Id"]
        material_count = building[f"Material{i}_Count"] or 0
        if pd.isna(material_id) or material_count == 0:
            continue
        building_rows.append({
            "building_id": building_id,
            "material_id_raw": material_id,
            "material_id": resolve_item_id(material_id),
        })

building_materials_long = pd.DataFrame(building_rows)
orphans_buildings = building_materials_long[building_materials_long["material_id"].isna()]
print(f"{len(orphans_buildings)} referência(s) de building que não resolvem para nenhum item conhecido:")
orphans_buildings


0 referência(s) de building que não resolvem para nenhum item conhecido:


,building_id,material_id_raw,material_id


## Regressão: referências penduradas no output já commitado

Mesma checagem, mas em cima de `src/data/items.json`/`buildings.json` já
gerados - garante que nada que chega no app tem uma referência sem destino.


In [6]:
committed_items = json.loads((REPO_ROOT / "src/data/items.json").read_text(encoding="utf-8"))
committed_buildings = json.loads((REPO_ROOT / "src/data/buildings.json").read_text(encoding="utf-8"))
known_ids = set(committed_items) | set(committed_buildings)

dangling = []
for source_name, db in [("items.json", committed_items), ("buildings.json", committed_buildings)]:
    for entry_id, entry in db.items():
        for ingredient_id in entry.get("ingredients", {}):
            if ingredient_id not in known_ids:
                dangling.append((source_name, entry_id, ingredient_id))

print(f"{len(dangling)} referência(s) pendurada(s) no output final já commitado:")
dangling


0 referência(s) pendurada(s) no output final já commitado:


[]